In [1]:
!pip install -q langchain-text-splitters

In [20]:
nltk.download('punkt_tab')

[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.


True

In [26]:
from langchain_text_splitters import (
    CharacterTextSplitter,
    RecursiveCharacterTextSplitter,
    MarkdownHeaderTextSplitter,
    NLTKTextSplitter
)
from pathlib import Path
import re
import nltk

import pandas as pd

In [8]:

PASTA = Path("/content/sample_data/mds")

arquivos = list(PASTA.glob("**/*.md"))

#print(f"Arquivos encontrados: {len(arquivos)}")

documentos = []

for arquivo in arquivos:
    texto = arquivo.read_text(encoding="utf-8")

    documentos.append({
        "arquivo": arquivo.name,
        "caminho": str(arquivo),
        #"texto": texto
    })

print(documentos)

# passa pelos arquivos
for item in documentos:
    print("Arquivo:", item['arquivo'])
    print("Caminho:", item['caminho'])

[{'arquivo': 'bert_pretraining.md', 'caminho': '/content/sample_data/mds/bert_pretraining.md'}, {'arquivo': 'attention_is_all_you_need.md', 'caminho': '/content/sample_data/mds/attention_is_all_you_need.md'}]
Arquivo: bert_pretraining.md
Caminho: /content/sample_data/mds/bert_pretraining.md
Arquivo: attention_is_all_you_need.md
Caminho: /content/sample_data/mds/attention_is_all_you_need.md


In [44]:
def chunk_calc(text, tamanho, overlap=0):
    splitter = CharacterTextSplitter(
        separator="",
        chunk_size=tamanho,
        chunk_overlap=overlap
    )

    chunks = splitter.split_text(text)
    return chunks


# Teste 7 - Por parágrafo

In [72]:
# ! Porém, há uma pegadinha: RecursiveCharacterTextSplitter ainda respeita chunk_size. Então isso não é exatamente um "splitter de parágrafos puro" se um parágrafo for muito grande.
# Teste 7
def analise_paragrafos(text, arquivo):
    splitter = RecursiveCharacterTextSplitter(
        chunk_size=999999,
        chunk_overlap=0,
        separators=["\n\n"]
    )


    chunks = splitter.create_documents([text])

    tamanhos = [
        len(chunk.page_content)
        for chunk in chunks
    ]

    metadados = {
        "arquivo": arquivo.name,
        "caminho": str(arquivo),
        "teste": 7,
        "estrategia": "Por parágrafo"
    }

    chunks = splitter.create_documents(
        [texto],
        metadatas=[metadados]
    )

    return {
        "Qtd. chunks": len(chunks),
        "Tamanho médio": round(sum(tamanhos) / len(tamanhos), 2),
        "Tamanho mínimo": min(tamanhos),
        "Tamanho máximo": max(tamanhos)
    }

analise = analise_paragrafos(texto, arquivo)

print(analise)

{'Qtd. chunks': 1, 'Tamanho médio': 70235.0, 'Tamanho mínimo': 70235, 'Tamanho máximo': 70235}


In [77]:
resultados_teste7 = []

for arquivo in arquivos:

    texto = arquivo.read_text(encoding="utf-8")

    analise = analise_paragrafos(
        texto,
        arquivo
    )

    resultados_teste7.append({
        "Arquivo": arquivo.name,
        **analise
    })

In [78]:
df_teste7 = pd.DataFrame(resultados_teste7)

display(df_teste7)

,Arquivo,Qtd. chunks,Tamanho médio,Tamanho mínimo,Tamanho máximo
0,bert_pretraining.md,1,70235.0,70235,70235
1,attention_is_all_you_need.md,1,48957.0,48957,48957


# Teste 8 - Sentenças agrupadas

In [95]:
# Teste 8

def analise_sentencas(texto, arquivo):

    splitter = NLTKTextSplitter()

    sentencas = splitter.split_text(texto)

    chunks = []

    for i in range(0, len(sentencas), 3):

        grupo = sentencas[i:i + 3]

        texto_chunk = " ".join(grupo)

        chunks.append({
            "texto": texto_chunk,
            "quantidade_sentencas": len(grupo),
            "metadados": {
                "arquivo": arquivo.name,
                "caminho": str(arquivo),
                "teste": 8,
                "estrategia": "Sentenças agrupadas",
                "sentencas": len(grupo)
            }
        })

    return chunks


In [82]:
def analisar_teste_8(chunks):

    tamanhos = [
        len(chunk["texto"])
        for chunk in chunks
    ]

    return {
        "Qtd. chunks": len(chunks),
        "Tamanho médio": round(sum(tamanhos) / len(tamanhos), 2),
        "Tamanho mínimo": min(tamanhos),
        "Tamanho máximo": max(tamanhos)
    }

In [96]:
resultados_teste8 = []

for arquivo in arquivos:

    texto = arquivo.read_text(encoding="utf-8")

    chunks = analise_sentencas(
        texto,
        arquivo
    )

    analise = analisar_teste_8(chunks)

    resultados_teste8.append({
        "Arquivo": arquivo.name,
        "Teste": 8,
        "Estratégia": "Sentenças agrupadas",
        "Configuração": "3 sentenças por chunk",
        **analise
    })

In [97]:
import pandas as pd

df_teste8 = pd.DataFrame(resultados_teste8)

display(df_teste8)

,Arquivo,Teste,Estratégia,Configuração,Qtd. chunks,Tamanho médio,Tamanho mínimo,Tamanho máximo
0,bert_pretraining.md,8,Sentenças agrupadas,3 sentenças por chunk,7,10490.71,4432,11868
1,attention_is_all_you_need.md,8,Sentenças agrupadas,3 sentenças por chunk,5,10086.60,1620,14045


In [98]:
for arquivo in arquivos:

    texto = arquivo.read_text(encoding="utf-8")

    chunks = analise_sentencas(
        texto,
        arquivo
    )

    print(f"\n\n{'=' * 60}")
    print(f"ARQUIVO: {arquivo.name}")
    print(f"{'=' * 60}")

    for i, chunk in enumerate(chunks[:3], start=1):

        print(f"\n--- Chunk {i} ---")
        print(chunk["texto"])



ARQUIVO: bert_pretraining.md

--- Chunk 1 ---
## BERT: Pre-training of Deep Bidirectional Transformers for Language Understanding

Jacob Devlin Ming-Wei Chang Kenton Lee Kristina Toutanova

Google AI Language

{ jacobdevlin,mingweichang,kentonl,kristout } @google.com

## Abstract

We introduce a new language representation model called BERT , which stands for B idirectional E ncoder R epresentations from T ransformers.

Unlike recent language representation models (Peters et al., 2018a; Radford et al., 2018), BERT is designed to pretrain deep bidirectional representations from unlabeled text by jointly conditioning on both left and right context in all layers.

As a result, the pre-trained BERT model can be finetuned with just one additional output layer to create state-of-the-art models for a wide range of tasks, such as question answering and language inference, without substantial taskspecific architecture modifications.

BERT is conceptually simple and empirically powerful.

It o


# Teste 9: Recursivo (separadores hierárquicos)




In [99]:
# # Teste 9: Recursivo (separadores hierárquicos)
# def analisar_recursivo(text):
#   sp9 = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=50, separators=["\n\n", "\n", " ", ""])
#   #print("Quantidade de chunks recursivos:", len(sp9.split_text(text)))

#   return sp9

#   from langchain_text_splitters import RecursiveCharacterTextSplitter


def analise_recursivo(texto, arquivo):

    splitter = RecursiveCharacterTextSplitter(
        chunk_size=500,
        chunk_overlap=50,
        separators=[
            "\n\n",
            "\n",
            " ",
            ""
        ]
    )

    documentos = splitter.create_documents(
        [texto],
        metadatas=[{
            "arquivo": arquivo.name,
            "caminho": str(arquivo),
            "teste": 9,
            "estrategia": "Recursive Character",
            "chunk_size": 500,
            "chunk_overlap": 50
        }]
    )

    return documentos

In [90]:
def analisar_chunks_documentos(chunks):

    tamanhos = [
        len(chunk.page_content)
        for chunk in chunks
    ]

    return {
        "Qtd. chunks": len(chunks),
        "Tamanho médio": round(sum(tamanhos) / len(tamanhos), 2),
        "Tamanho mínimo": min(tamanhos),
        "Tamanho máximo": max(tamanhos)
    }

In [100]:
chunks_teste9 = analise_recursivo(
    texto,
    arquivo
)

print("Quantidade de chunks:", len(chunks_teste9))

for i, chunk in enumerate(chunks_teste9[:3], start=1):

    print(f"\n===== CHUNK {i} =====")
    print(chunk.page_content)

    print("\nMetadados:")
    print(chunk.metadata)

    analise = analisar_chunks_documentos(chunks_teste9)

    print(analise)

Quantidade de chunks: 137

===== CHUNK 1 =====
Provided proper attribution is provided, Google hereby grants permission to reproduce the tables and figures in this paper solely for use in journalistic or scholarly works.

## Attention Is All You Need

Ashish Vaswani ∗ Google Brain avaswani@google.com Noam Shazeer ∗ Google Brain noam@google.com

Metadados:
{'arquivo': 'attention_is_all_you_need.md', 'caminho': '/content/sample_data/mds/attention_is_all_you_need.md', 'teste': 9, 'estrategia': 'Recursive Character', 'chunk_size': 500, 'chunk_overlap': 50}
{'Qtd. chunks': 137, 'Tamanho médio': 364.13, 'Tamanho mínimo': 26, 'Tamanho máximo': 499}

===== CHUNK 2 =====
Llion Jones ∗ Google Research llion@google.com Niki Parmar ∗ Google Research nikip@google.com Aidan N. Gomez ∗ † University of Toronto aidan@cs.toronto.edu Jakob Uszkoreit ∗ Google Research usz@google.com Łukasz Kaiser ∗ Google Brain lukaszkaiser@google.com Illia Polosukhin ∗ ‡

illia.polosukhin@gmail.com

## Abstract

Metadado

In [101]:
resultados_teste9 = []

for arquivo in arquivos:

    texto = arquivo.read_text(encoding="utf-8")

    chunks = analise_recursivo(
        texto,
        arquivo
    )

    analise = analisar_chunks_documentos(chunks)

    resultados_teste9.append({
        "Arquivo": arquivo.name,
        "Teste": 9,
        "Estratégia": "Recursive Character",
        "Configuração": "500 caracteres, overlap 50",
        **analise
    })

df_teste9 = pd.DataFrame(resultados_teste9)

display(df_teste9)

,Arquivo,Teste,Estratégia,Configuração,Qtd. chunks,Tamanho médio,Tamanho mínimo,Tamanho máximo
0,bert_pretraining.md,9,Recursive Character,"500 caracteres, overlap 50",201,356.39,9,499
1,attention_is_all_you_need.md,9,Recursive Character,"500 caracteres, overlap 50",137,364.13,26,499


# Teste 10: Por seção / heading do Markdown

In [103]:
# # Teste 10: Por seção / heading do Markdown
# def analisar_markdown(text):
#     headers = [("#", "Header 1"), ("##", "Header 2"), ("###", "Header 3")]
#     sp10 = MarkdownHeaderTextSplitter(headers_to_split_on=headers)
#     docs_md = sp10.split_text(text)
#     chunks_md = [doc.page_content for doc in docs_md]

#     #print("Quantidade de chunks por seção / heading do Markdown:", len(chunks_md))

#     return chunks_md

from langchain_text_splitters import MarkdownHeaderTextSplitter


def teste_10_markdown(texto, arquivo):

    headers = [
        ("#", "Header 1"),
        ("##", "Header 2"),
        ("###", "Header 3")
    ]

    splitter = MarkdownHeaderTextSplitter(
        headers_to_split_on=headers,
        strip_headers=False
    )

    documentos = splitter.split_text(texto)

    # Adicionar informações do arquivo/teste
    for doc in documentos:
        doc.metadata["arquivo"] = arquivo.name
        doc.metadata["caminho"] = str(arquivo)
        doc.metadata["teste"] = 10
        doc.metadata["estrategia"] = "Markdown / estrutura semântica"

    return documentos

In [104]:
def analisar_chunks_documentos(chunks):

    tamanhos = [
        len(chunk.page_content)
        for chunk in chunks
    ]

    return {
        "Qtd. chunks": len(chunks),
        "Tamanho médio": round(sum(tamanhos) / len(tamanhos), 2),
        "Tamanho mínimo": min(tamanhos),
        "Tamanho máximo": max(tamanhos)
    }

In [106]:
resultados_teste10 = []

for arquivo in arquivos:

    texto = arquivo.read_text(encoding="utf-8")

    chunks = teste_10_markdown(
        texto,
        arquivo
    )

    analise = analisar_chunks_documentos(chunks)

    resultados_teste10.append({
        "Arquivo": arquivo.name,
        "Teste": 10,
        "Estratégia": "Markdown / estrutura semântica",
        "Configuração": "#, ##, ###",
        **analise
    })

df_teste10 = pd.DataFrame(resultados_teste10)

display(df_teste10)

,Arquivo,Teste,Estratégia,Configuração,Qtd. chunks,Tamanho médio,Tamanho mínimo,Tamanho máximo
0,bert_pretraining.md,10,Markdown / estrutura semântica,"#, ##, ###",33,2131.33,32,11380
1,attention_is_all_you_need.md,10,Markdown / estrutura semântica,"#, ##, ###",28,1750.11,12,10946


In [51]:
nltk_splitter = NLTKTextSplitter()

def analise_sentencas(text):

    splitter = NLTKTextSplitter()

    sentencas = splitter.split_text(text)

    chunks = [
        " ".join(sentencas[i:i+3])
        for i in range(0, len(sentencas), 3)
    ]

    return chunks

In [102]:
# não sei se precisa, mas apra analisar quantas partes do chunk existem dentro de outros chunks
def contar_chunks_sobrepostos(chunks):

    contador = 0

    for i in range(1, len(chunks)):

        chunk_anterior = chunks[i - 1]
        chunk_atual = chunks[i]

        tamanho_maximo = min(
            len(chunk_anterior),
            len(chunk_atual)
        )

        possui_overlap = False

        for tamanho in range(1, tamanho_maximo + 1):

            if chunk_anterior[-tamanho:] == chunk_atual[:tamanho]:
                possui_overlap = True
                break

        if possui_overlap:
            contador += 1

    return contador

# Analiser os chunks

In [53]:
def analisar_chunks(chunks, chunk_size, chunk_overlap):

    tamanhos = [len(chunk) for chunk in chunks]

    # Quantidade de chunks que possuem overlap
    if chunk_overlap > 0:
        qtd_sobrepostos = contar_chunks_sobrepostos(chunks)
        #qtd_sobrepostos = max(len(chunks) - 1, 0)
    else:
        qtd_sobrepostos = 0

    # Percentual configurado de overlap
    percentual_overlap = (
        chunk_overlap / chunk_size * 100
        if chunk_size > 0
        else 0
    )



    # aqui dá para comparar com oque o embedding vai realmente retornar depois!
    # Estimativa simples:
    # aproximadamente 4 caracteres por token
    total_caracteres = sum(tamanhos)
    tokens_estimados = round(total_caracteres / 4)

    return {
        "Qtd. chunks": len(chunks),
        "Tamanho médio": round(sum(tamanhos) / len(tamanhos), 2),
        "Tamanho mínimo": min(tamanhos),
        "Tamanho máximo": max(tamanhos),
        "Chunks sobrepostos": qtd_sobrepostos,
        "Overlap (%)": round(percentual_overlap, 2),
        "Tokens estimados": tokens_estimados
    }

In [50]:
# Aqui define os testes a serem realizados
testes = [
    {
        "Teste": 1,
        "Estratégia": "Fixo",
        "Configuração": "200 caracteres, overlap 0",
        "chunk_size": 200,
        "chunk_overlap": 0
    },
    {
        "Teste": 2,
        "Estratégia": "Fixo",
        "Configuração": "500 caracteres, overlap 0",
        "chunk_size": 500,
        "chunk_overlap": 0
    },
    {
        "Teste": 3,
        "Estratégia": "Fixo",
        "Configuração": "1000 caracteres, overlap 0",
        "chunk_size": 1000,
        "chunk_overlap": 0
    },
    {
        "Teste": 4,
        "Estratégia": "Fixo",
        "Configuração": "2000 caracteres, overlap 0",
        "chunk_size": 2000,
        "chunk_overlap": 0
    },
    {
        "Teste": 5,
        "Estratégia": "Fixo + overlap",
        "Configuração": "500 caracteres, overlap 50",
        "chunk_size": 500,
        "chunk_overlap": 50
    },
    {
        "Teste": 6,
        "Estratégia": "Fixo + overlap",
        "Configuração": "500 caracteres, overlap 200",
        "chunk_size": 500,
        "chunk_overlap": 200
    }
]

In [54]:
# Aqui é onde as coisas são executadas
# Ele pega os arquivos com o PATH definido lá em cima
# Faz um loop e manda calcular e depois analizar os chunks para retornar oque é pedido
# Ele vai guardando cada resultado dentro de resultados = []

resultados = []

for arquivo in arquivos:

    texto = arquivo.read_text(encoding="utf-8")

    for teste in testes:

        chunks = chunk_calc(
            texto,
            teste["chunk_size"],
            teste["chunk_overlap"]
        )

        analise = analisar_chunks(
            chunks,
            teste["chunk_size"],
            teste["chunk_overlap"]
        )

        resultados.append({
            "Arquivo": arquivo.name,
            "Teste": teste["Teste"],
            "Estratégia": teste["Estratégia"],
            "Configuração": teste["Configuração"],
            **analise
        })

# Cria a tabela
df_resultados = pd.DataFrame(resultados)

display(df_resultados)

,Arquivo,Teste,Estratégia,Configuração,Qtd. chunks,Tamanho médio,Tamanho mínimo,Tamanho máximo,Chunks sobrepostos,Overlap (%),Tokens estimados
0,bert_pretraining.md,1,Fixo,"200 caracteres, overlap 0",352,197.32,35,200,0,0.0,17364
1,bert_pretraining.md,2,Fixo,"500 caracteres, overlap 0",141,496.12,234,500,0,0.0,17488
2,bert_pretraining.md,3,Fixo,"1000 caracteres, overlap 0",71,987.75,234,1000,0,0.0,17532
3,bert_pretraining.md,4,Fixo,"2000 caracteres, overlap 0",36,1950.22,234,2000,0,0.0,17552
4,bert_pretraining.md,5,Fixo + overlap,"500 caracteres, overlap 50",156,497.53,440,500,155,10.0,19404
5,bert_pretraining.md,6,Fixo + overlap,"500 caracteres, overlap 200",234,497.38,335,500,233,40.0,29097
6,attention_is_all_you_need.md,1,Fixo,"200 caracteres, overlap 0",245,192.89,117,200,0,0.0,11814
7,attention_is_all_you_need.md,2,Fixo,"500 caracteres, overlap 0",98,492.66,425,500,0,0.0,12070
8,attention_is_all_you_need.md,3,Fixo,"1000 caracteres, overlap 0",49,993.20,937,1000,0,0.0,12167
9,attention_is_all_you_need.md,4,Fixo,"2000 caracteres, overlap 0",25,1951.04,956,2000,0,0.0,12194


In [39]:
# TESTE, esta parte é puramente para visualizar e testar como as coisas estavm retornado

for item in documentos:
    item_to_anylise = item['caminho']
    print(item_to_anylise)
    with open(item_to_anylise,"r",encoding="utf-8") as file:
        text = file.read()

    teste1 = chunk_calc(text, 200, 0)
    teste2 = chunk_calc(text, 500, 0)
    teste3 = chunk_calc(text, 1000, 0)
    teste4 = chunk_calc(text, 2000, 0)

    teste5 = chunk_calc(text, 500, 50)
    teste6 = chunk_calc(text, 500, 200)

    test_results_to_analyze = [teste1, teste2, teste3, teste4, teste5, teste6]

    print(f"Tamnho do Chunk teste1: {len(teste1)}")
    print(f"Tamanho do Chunk teste2: {len(teste2)}")
    print(f"Tamanho do Chunk teste3: {len(teste3)}")
    print(f"Tamanho do Chunk teste4: {len(teste4)}")
    print(f"Tamanho do Chunk teste5: {len(teste5)}")
    print(f"Tamanho do Chunk teste6: {len(teste6)}")

    #teste 8
    sentencas = nltk_splitter.split_text(text)
    print(f"Tamanho Sentenças {len(sentencas)}")

    teste9 = analisar_recursivo(text)
    print("Quantidade de chunks recursivos:", len(teste9.split_text(text)))

    teste10 = analisar_markdown(text)
    print("Quantidade de chunks por seção / heading do Markdown:", len(teste10))


/content/sample_data/mds/bert_pretraining.md
Tamnho do Chunk teste1: 352
Tamanho do Chunk teste2: 141
Tamanho do Chunk teste3: 71
Tamanho do Chunk teste4: 36
Tamanho do Chunk teste5: 156
Tamanho do Chunk teste6: 234
Tamanho Sentenças 20
Quantidade de chunks recursivos: 201
Quantidade de chunks por seção / heading do Markdown: 30
/content/sample_data/mds/attention_is_all_you_need.md


Tamnho do Chunk teste1: 245
Tamanho do Chunk teste2: 98
Tamanho do Chunk teste3: 49
Tamanho do Chunk teste4: 25
Tamanho do Chunk teste5: 109
Tamanho do Chunk teste6: 163
Tamanho Sentenças 13
Quantidade de chunks recursivos: 137
Quantidade de chunks por seção / heading do Markdown: 27
